In [ ]:
import requests
from bs4 import BeautifulSoup
from pathlib import Path


# =========================
# PARAMÈTRES
# =========================

YEAR = 2026

# Jours de l'année à télécharger
DAYS = list(range(201,203))

# Dossier principal
DATA_DIR = Path("data")

#ne retélécharge pas un fichier déjà présent
OVERWRITE = False

In [ ]:
# =========================
# 1. RÉCUPÉRER LES FICHIERS DISPONIBLES POUR UN JOUR
# =========================

def get_day_files(year, doy):

    doy_str = f"{doy:03d}"
    year_short = year % 100

    base_url = f"https://rgpdata.ign.fr/pub/data/{year}/{doy_str}/"

    response = requests.get(base_url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    links = [
        link["href"]
        for link in soup.find_all("a", href=True)
    ]


    # -------------------------
    # XML des stations
    # -------------------------

    xml_suffix = f"{doy_str}.{year_short:02d}.xml"

    xml_files = sorted([
        filename
        for filename in links
        if filename.lower().endswith(xml_suffix.lower())
    ])


    # -------------------------
    # Métadonnées des stations
    # -------------------------

    metadata_suffix = f"{year}-{doy_str}.txt"

    metadata_files = sorted([
        filename
        for filename in links
        if filename.lower().startswith("coor")
        and filename.lower().endswith(metadata_suffix.lower())
    ])


    return base_url, xml_files, metadata_files

In [ ]:
# =========================
# 2. FONCTION DE TÉLÉCHARGEMENT
# =========================

def download_file(file_url, destination, overwrite=False):

    # Ne pas retélécharger si le fichier existe déjà
    if destination.exists() and not overwrite:
        return "skipped"

    response = requests.get(file_url, timeout=30)
    response.raise_for_status()

    destination.write_bytes(response.content)

    return "downloaded"

In [ ]:
# =========================
# 3. TÉLÉCHARGER LES DONNÉES
# =========================

for doy in DAYS:

    doy_str = f"{doy:03d}"

    print("\n" + "=" * 70)
    print(f"ANNÉE {YEAR} - JOUR {doy_str}")
    print("=" * 70)


    # -------------------------
    # Dossier local
    # -------------------------

    output_dir = DATA_DIR / str(YEAR) / doy_str
    output_dir.mkdir(parents=True, exist_ok=True)


    # -------------------------
    # Récupérer la liste
    # -------------------------

    try:

        base_url, xml_files, metadata_files = get_day_files(
            YEAR,
            doy
        )

    except requests.RequestException as e:

        print(f"Impossible d'accéder au jour {doy_str} : {e}")
        continue


    print(f"{len(xml_files)} XML de stations trouvés.")
    print(f"{len(metadata_files)} fichier(s) de métadonnées trouvé(s).")


    # -------------------------
    # Télécharger les XML
    # -------------------------

    downloaded_xml = 0
    skipped_xml = 0

    for filename in xml_files:

        status = download_file(
            file_url=base_url + filename,
            destination=output_dir / filename,
            overwrite=OVERWRITE
        )

        if status == "downloaded":
            downloaded_xml += 1
        else:
            skipped_xml += 1


    # -------------------------
    # Télécharger les métadonnées
    # -------------------------

    downloaded_metadata = 0
    skipped_metadata = 0

    for filename in metadata_files:

        status = download_file(
            file_url=base_url + filename,
            destination=output_dir / filename,
            overwrite=OVERWRITE
        )

        if status == "downloaded":
            downloaded_metadata += 1
        else:
            skipped_metadata += 1


    # -------------------------
    # Résumé
    # -------------------------

    print(
        f"XML : {downloaded_xml} téléchargé(s), "
        f"{skipped_xml} déjà présent(s)"
    )

    print(
        f"Métadonnées : {downloaded_metadata} téléchargé(s), "
        f"{skipped_metadata} déjà présent(s)"
    )